# 📊 City Air Quality Report

A full-year analysis of **NO₂ and PM2.5** over Paris for 2023.  
We will look at monthly trends, weekday patterns, and compare both pollutants side by side.

**Requirements**
```
pip install jiskta matplotlib seaborn pandas numpy
```

In [ ]:
# Install the SDK from the private GitHub repo.
# Generate a read-only token at: https://github.com/settings/tokens/new
#   → select scope: Contents (read-only)
# Then run (replace TOKEN with your actual token):
# !pip install "git+https://TOKEN@github.com/fvsever/jiskta-python.git#egg=jiskta[pandas]" matplotlib seaborn numpy

In [ ]:
from jiskta import JisktaClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

API_KEY = "sk_live_YOUR_API_KEY"
client = JisktaClient(api_key=API_KEY)

## 1 — Download full year of daily data (NO₂ + PM2.5)

In [ ]:
df = client.query(
    lat=(48.7, 49.0),
    lon=(2.2, 2.5),
    start="2023-01",
    end="2023-12",
    variables=["no2", "pm2p5"],
    aggregate="daily",
)

df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["dayofweek"] = df["date"].dt.day_name()

# Area-average over all grid points
daily = df.groupby("date")[["no2_mean", "pm2p5_mean"]].mean().reset_index()
daily["month"] = daily["date"].dt.month
daily["month_name"] = daily["date"].dt.strftime("%b")

print(f"{len(daily)} daily records  |  columns: {list(daily.columns)}")
daily.head()

## 2 — Annual overview: dual-axis time series

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()

ax1.plot(daily["date"], daily["no2_mean"], color="#1a6bbf", linewidth=1.5, label="NO₂")
ax1.fill_between(daily["date"], daily["no2_mean"], alpha=0.12, color="#1a6bbf")
ax2.plot(daily["date"], daily["pm2p5_mean"], color="#e07b39", linewidth=1.5, linestyle="--", label="PM2.5")

ax1.set_ylabel("NO₂  (µg/m³)", color="#1a6bbf")
ax2.set_ylabel("PM2.5  (µg/m³)", color="#e07b39")
ax1.tick_params(axis="y", labelcolor="#1a6bbf")
ax2.tick_params(axis="y", labelcolor="#e07b39")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax1.xaxis.set_major_locator(mdates.MonthLocator())

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

ax1.set_title("Daily NO₂ and PM2.5 — Paris, 2023", fontsize=14, pad=12)
ax1.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 3 — Monthly box plots

In [ ]:
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, col, color, title in [
    (axes[0], "no2_mean",   "#1a6bbf", "NO₂"),
    (axes[1], "pm2p5_mean", "#e07b39", "PM2.5"),
]:
    data = [daily.loc[daily["month_name"] == m, col].dropna().values for m in month_order]
    bp = ax.boxplot(data, labels=month_order, patch_artist=True, medianprops=dict(color="white", linewidth=2))
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_title(f"Monthly distribution — {title}  (µg/m³)", fontsize=12)
    ax.set_xlabel("")

fig.tight_layout()
plt.show()

## 4 — Day-of-week pattern

Do weekdays have higher NO₂ than weekends due to traffic?

In [ ]:
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df_dow = df.groupby("dayofweek")[["no2_mean","pm2p5_mean"]].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(dow_order))
w = 0.35
ax.bar(x - w/2, df_dow["no2_mean"],   w, label="NO₂",   color="#1a6bbf", alpha=0.8)
ax.bar(x + w/2, df_dow["pm2p5_mean"], w, label="PM2.5", color="#e07b39", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([d[:3] for d in dow_order])
ax.set_ylabel("Mean concentration  (µg/m³)")
ax.set_title("Average pollutant level by day of week — Paris, 2023", fontsize=13, pad=10)
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 5 — Annual summary table

In [ ]:
summary = daily[["no2_mean","pm2p5_mean"]].agg(["min","mean","max","std"]).T
summary.columns = ["Min", "Mean", "Max", "Std Dev"]
summary.index = ["NO₂ (µg/m³)", "PM2.5 (µg/m³)"]
summary.round(2)

---
**Next:** See [03_who_exceedance.ipynb](03_who_exceedance.ipynb) to identify days that exceed WHO guidelines.